# Sampling (rejection, importance) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Estimate expectations by drawing from an easier distribution and correcting the mismatch
Sampling methods replace analytic sums with random draws. These examples compute rejection acceptance, importance weights, self-normalized estimates, weight degeneracy, and proposal quality.

In [ ]:
# 1) rejection sampling acceptance rate for target under envelope Mq
t=np.array([0.1,0.2,0.4,0.3]); q=np.ones(4)/4; M=np.max(t/q); accept=1/M
plt.figure(figsize=(6,3)); plt.bar(['acceptance rate'],[accept]); plt.title(f'M={M:.1f}, accept={accept:.3f}')
assert abs(accept-0.625)<1e-12

In [ ]:
# 2) importance weights are target/proposal ratios
w=t/q; plt.figure(figsize=(6,3)); plt.bar(range(4),w); plt.title('importance weights t/q')
assert abs(w.max()-1.6)<1e-12

In [ ]:
# 3) self-normalized importance estimate of E[X]
x=np.arange(4); est=np.sum((w/w.sum())*x); true=np.sum(t*x)
plt.figure(figsize=(6,3)); plt.bar(['importance','true'],[est,true]); plt.title('weighted average recovers expectation')
assert abs(est-true)<1e-12 and abs(true-1.9)<1e-12

In [ ]:
# 4) bad proposal causes low effective sample size
qbad=np.array([0.7,0.1,0.1,0.1]); wb=t/qbad; ess=(wb.sum()**2)/(wb@wb)
plt.figure(figsize=(6,3)); plt.bar(['ESS','4 samples'],[ess,4]); plt.title('uneven weights waste samples')
assert ess<3

In [ ]:
# 5) closer proposal improves ESS
qgood=np.array([0.15,0.2,0.35,0.3]); wg=t/qgood; essg=(wg.sum()**2)/(wg@wg)
plt.figure(figsize=(6,3)); plt.bar(['bad q','good q'],[ess,essg]); plt.title('proposal quality matters')
assert essg>ess